# Openness and Closeness in Seabed Morphological Mapping

**Context:** This notebook explains the concepts of *topographic openness* and *closeness* as used in the GA-SaMMT (Geoscience Australia's Semi-automated Morphological Mapping Tools) for characterising seabed features from bathymetric data.

**References:**
- Yokoyama, R., Shirasawa, M., & Pike, R. J. (2002). Visualizing topography by openness: A new application of image processing to digital elevation models. *Photogrammetric Engineering and Remote Sensing*, 68(3), 257–265.
- Huang, Z., Nanson, R., McNeil, M., Wenderlich, M., Gafeira, J., Post, A, Nichol, S. (2023). Rule-based semi-automated tools for mapping seabed morphology from bathymetry data. *Frontiers in Marine Science*, 10, 1236788.

---

## 1. What is Topographic Openness?

**Topographic openness** is a terrain attribute that quantifies how *open* or *enclosed* a location is relative to its surrounding landscape. It was introduced by Yokoyama et al. (2002) originally for land topography, and has since been adapted for bathymetric (underwater) terrain analysis.

The concept is based on **zenith angles** — angles measured upward from a horizontal plane to the terrain surface along radial lines emanating from a central point.

For a given point on the terrain:
- We cast rays outward in multiple directions (typically 8 principal directions: N, NE, E, SE, S, SW, W, NW).
- Along each ray, we find the **maximum zenith angle** to the horizon (the steepest upward angle we can "see").
- We average these angles across all directions.

There are **two complementary measures**:

| Measure | Symbol | Intuition |
|---|---|---|
| **Positive Openness (PO)** | Ψ⁺ | How open is the sky above? High = open terrain (ridges, hills) |
| **Negative Openness (NO)** | Ψ⁻ | How enclosed is the terrain below? High = enclosed terrain (valleys, pits) |

## 2. Mathematical Definition

For a point $P$ on a digital elevation model (DEM) or bathymetric grid, consider $n$ radial directions (usually $n = 8$). For each direction $i$, we search along a line of length $L$ (the **search radius** in cell units).

### Positive Openness

For each direction $i$, find the **maximum elevation angle** $\alpha_i$ to any point along that radial line:

$$\alpha_i = \max_{d=1}^{L} \arctan\left(\frac{z(d) - z(P)}{d \cdot \Delta x}\right)$$

where:
- $z(d)$ = elevation at distance $d$ along the radial line
- $z(P)$ = elevation at the central point
- $\Delta x$ = grid cell size

The **Positive Openness** is:

$$\Psi^+ = \frac{1}{n} \sum_{i=1}^{n} \left(\frac{\pi}{2} - \alpha_i\right)$$

This is the mean **zenith angle** — $90°$ minus the maximum elevation angle in each direction.

### Negative Openness

For each direction $i$, find the **minimum depression angle** $\beta_i$ (looking downward):

$$\beta_i = \max_{d=1}^{L} \arctan\left(\frac{z(P) - z(d)}{d \cdot \Delta x}\right)$$

The **Negative Openness** is:

$$\Psi^- = \frac{1}{n} \sum_{i=1}^{n} \left(\frac{\pi}{2} - \beta_i\right)$$

> **Key insight:** Both openness values range from 0 to π/2 radians (0° to 90°). **Higher positive openness** → more exposed / convex terrain. **Lower negative openness** (or equivalently, higher closeness) → more enclosed / concave terrain.

## 3. Visual Intuition

The diagram below illustrates the zenith angle concept for a 2D cross-section. Imagine standing at point P and looking along a radial line:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Topographic Openness — 2D Cross-section Illustration", fontsize=14, fontweight='bold')

# ---- LEFT: Positive Openness on a RIDGE ----
ax = axes[0]
x = np.linspace(0, 10, 200)
# A ridge (Gaussian bump)
z = 2 * np.exp(-0.5 * ((x - 5) / 1.2) ** 2)
ax.fill_between(x, z, alpha=0.3, color='steelblue', label='Seabed profile')
ax.plot(x, z, color='steelblue', lw=2)

# Central point P (at the ridge peak)
p_x, p_z = 5.0, float(2 * np.exp(-0.5 * ((5 - 5) / 1.2) ** 2))
ax.plot(p_x, p_z, 'ro', ms=10, zorder=5, label='Point P (ridge top)')

# Horizontal reference line
ax.axhline(p_z, color='gray', ls='--', lw=1, label='Horizontal plane')

# Radial ray to the right — the horizon is low, zenith angle is large
r_x = 8.5
r_z = float(2 * np.exp(-0.5 * ((r_x - 5) / 1.2) ** 2))
ax.annotate('', xy=(r_x, r_z), xytext=(p_x, p_z),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(7.5, p_z + 0.15, r'Small $\alpha$ → large $\Psi^+$', color='red', fontsize=9)

ax.set_xlim(0, 10)
ax.set_ylim(-0.2, 2.8)
ax.set_title("Positive Openness — Ridge (Bathymetric High)\nHigh Ψ⁺: terrain is open / convex", fontsize=11)
ax.set_xlabel("Horizontal distance")
ax.set_ylabel("Elevation / Depth (inverted for seabed)")
ax.legend(fontsize=8)

# ---- RIGHT: Negative Openness on a VALLEY ----
ax2 = axes[1]
# A valley (inverted Gaussian)
z2 = -2 * np.exp(-0.5 * ((x - 5) / 1.2) ** 2) + 2
ax2.fill_between(x, z2, 2.3, alpha=0.3, color='darkorange', label='Seabed profile')
ax2.plot(x, z2, color='darkorange', lw=2)

# Central point P (at the valley bottom)
p2_x, p2_z = 5.0, float(-2 * np.exp(0) + 2)
ax2.plot(p2_x, p2_z, 'ro', ms=10, zorder=5, label='Point P (valley floor)')
ax2.axhline(p2_z, color='gray', ls='--', lw=1, label='Horizontal plane')

# Radial ray to the right — the terrain rises sharply, zenith angle is small
r2_x = 8.0
r2_z = float(-2 * np.exp(-0.5 * ((r2_x - 5) / 1.2) ** 2) + 2)
ax2.annotate('', xy=(r2_x, r2_z), xytext=(p2_x, p2_z),
            arrowprops=dict(arrowstyle='->', color='darkred', lw=2))
ax2.text(6.0, p2_z + 0.3, r'Large $\beta$ → small $\Psi^-$', color='darkred', fontsize=9)

ax2.set_xlim(0, 10)
ax2.set_ylim(-0.2, 2.8)
ax2.set_title("Negative Openness — Valley / Pockmark (Bathymetric Low)\nLow Ψ⁻: terrain is enclosed / concave", fontsize=11)
ax2.set_xlabel("Horizontal distance")
ax2.set_ylabel("Elevation / Depth")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("openness_illustration.png", dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 4. Relationship Between Openness and Seabed Features

Because bathymetric data records **depth** (values increase downward), the openness measures behave in a specific way:

### Bathymetric Highs (seamounts, ridges, mounds, hummocks)

A bathymetric high is a location **shallower** than its surroundings — a "bump" in the seabed.

- **Negative Openness (NO)** is **small** at a bathymetric high.
  - Why? Looking outward from the top of a mound, the surrounding terrain drops away. The depression angles are large → negative openness is small.
  - In the GA-SaMMT openness tool, **negative openness is used to detect bathymetric highs** — regions with NO below a threshold are classified as high features.

### Bathymetric Lows (pockmarks, valleys, canyons, depressions)

A bathymetric low is a location **deeper** than its surroundings — a "pit" or "valley".

- **Positive Openness (PO)** is **small** at a bathymetric low.
  - Why? At the bottom of a pit, the surrounding terrain rises steeply. The zenith angles are small → positive openness is small.
  - In the GA-SaMMT openness tool, **positive openness is used to detect bathymetric lows** — regions with PO below a threshold are classified as low features.

Summary table:

| Feature type | Positive Openness (PO) | Negative Openness (NO) | Detection metric |
|---|---|---|---|
| **Bathymetric High** (ridge, mound) | High | **Low** | Low NO |
| **Flat seabed** | Medium | Medium | Neither |
| **Bathymetric Low** (pit, canyon) | **Low** | High | Low PO |

## 5. The Openness Circle Radius Parameter

The **search radius** $L$ (called *Openness Circle Radius* in GA-SaMMT, measured in grid cell units) controls the **spatial scale** of the features being detected.

- **Small radius** (e.g., 10 cells) → sensitive to **fine-scale** features (small hummocks, micro-pockmarks)
- **Large radius** (e.g., 100 cells) → sensitive to **broad-scale** features (seamounts, large canyons)

This is analogous to the TPI (Topographic Position Index) radius parameter — it sets the neighbourhood window over which "context" is evaluated.

**Practical tip:** Run the tool at multiple radii and merge results to capture features at different scales.

## 6. Thresholding: How Features are Extracted

Once the NO or PO raster is computed, GA-SaMMT applies a **threshold** to identify candidate feature cells:

For **Bathymetric Highs** (using NO):
$$\text{NO\_threshold} = \overline{\text{NO}} - c \cdot \sigma_{\text{NO}}$$

- Cells where $\text{NO} < \text{NO\_threshold}$ are candidate high-feature cells.
- Parameter $c$ corresponds to the **NO STD Scale Large** parameter in the tool (e.g., $c = 1.0$ means the threshold is 1 standard deviation below the mean).
- A second, smaller threshold (**NO STD Scale Small**, e.g., $c = 0$) is used to capture the "tops" of the features more precisely.

For **Bathymetric Lows** (using PO):
$$\text{PO\_threshold} = \overline{\text{PO}} - c \cdot \sigma_{\text{PO}}$$

- Cells where $\text{PO} < \text{PO\_threshold}$ are candidate low-feature cells.

After thresholding, a minimum **area threshold** is applied to remove noise polygons that are too small to represent meaningful features.

## 7. Comparison with TPI (Topographic Position Index)

GA-SaMMT offers both TPI-based and Openness-based methods. Understanding their differences helps choose the right tool:

| Property | TPI | Openness |
|---|---|---|
| **What it measures** | Difference between a cell's elevation and the mean of its neighbourhood | Angular openness of the terrain from a point |
| **Computation** | Fast (mean filter) | Slower (8-direction ray casting) |
| **Sensitivity** | Linear differences in depth | Angular shape of terrain — better for asymmetric or complex features |
| **Best for** | General morphology, large datasets | Features with strong local curvature (pockmarks, volcanic cones) |
| **Scale control** | Radius of neighbourhood window | Search radius for ray casting |

The two methods are **complementary** — in practice, running both and comparing results (or combining outputs) gives the most robust feature detection.

## 8. Coding Openness from Scratch

Below is a simple Python implementation of positive and negative openness on a 2D elevation grid. This is a **didactic** implementation — GA-SaMMT uses an ArcGIS raster function (`Openness Circle Radius`) that is highly optimised.

In [ ]:
import numpy as np

def compute_openness(dem: np.ndarray, radius: int, cell_size: float = 1.0, n_directions: int = 8):
    """
    Compute Positive Openness (PO) and Negative Openness (NO) for a 2D DEM.

    Parameters
    ----------
    dem        : 2D numpy array of elevations (or bathymetric depths)
    radius     : search radius in grid cells
    cell_size  : spatial resolution of the grid (metres per cell)
    n_directions: number of radial directions (8 is standard)

    Returns
    -------
    po : Positive Openness array (radians), same shape as dem
    no : Negative Openness array (radians), same shape as dem
    """
    rows, cols = dem.shape
    po = np.full_like(dem, np.nan, dtype=float)
    no = np.full_like(dem, np.nan, dtype=float)

    # Build unit direction vectors
    angles = np.linspace(0, 2 * np.pi, n_directions, endpoint=False)
    directions = [(np.cos(a), np.sin(a)) for a in angles]

    for r in range(rows):
        for c in range(cols):
            z_p = dem[r, c]
            max_elev_angles = []   # for positive openness
            max_depr_angles = []   # for negative openness

            for (dy, dx) in directions:
                max_elev = -np.inf   # maximum elevation angle seen
                max_depr = -np.inf   # maximum depression angle seen

                for d in range(1, radius + 1):
                    nr = int(round(r + d * dy))
                    nc = int(round(c + d * dx))
                    if nr < 0 or nr >= rows or nc < 0 or nc >= cols:
                        break  # ray exits the grid

                    z_q = dem[nr, nc]
                    dist = d * cell_size
                    dz = z_q - z_p

                    # Elevation angle (upward): positive when z_q > z_p
                    elev_angle = np.arctan(dz / dist)
                    # Depression angle (downward): positive when z_q < z_p
                    depr_angle = np.arctan(-dz / dist)

                    max_elev = max(max_elev, elev_angle)
                    max_depr = max(max_depr, depr_angle)

                max_elev_angles.append(max_elev)
                max_depr_angles.append(max_depr)

            # Positive openness = mean zenith angle = mean(π/2 - max_elev_angle)
            po[r, c] = np.mean([np.pi / 2 - a for a in max_elev_angles])
            # Negative openness = mean(π/2 - max_depr_angle)
            no[r, c] = np.mean([np.pi / 2 - a for a in max_depr_angles])

    return po, no

print("Function defined. Ready to test.")

In [ ]:
# ---------------------------------------------------------------
# Test on a synthetic bathymetric grid
# ---------------------------------------------------------------
np.random.seed(42)
size = 60
x_grid = np.linspace(-3, 3, size)
y_grid = np.linspace(-3, 3, size)
XX, YY = np.meshgrid(x_grid, y_grid)

# Base flat seabed
DEM = np.zeros((size, size))

# Add a bathymetric HIGH (seamount) — a Gaussian bump
DEM += 30 * np.exp(-0.5 * (XX**2 + YY**2) / 0.5**2)

# Add a bathymetric LOW (pockmark) — an inverted Gaussian
DEM -= 20 * np.exp(-0.5 * ((XX - 1.8)**2 + (YY - 1.8)**2) / 0.3**2)

# Add slight noise
DEM += np.random.normal(0, 0.5, DEM.shape)

print(f"DEM shape: {DEM.shape}")
print(f"DEM range: [{DEM.min():.1f}, {DEM.max():.1f}]")

# Compute openness with radius = 8 cells
RADIUS = 8
print(f"Computing openness with radius={RADIUS} cells... (may take ~30s for {size}x{size} grid)")
PO, NO = compute_openness(DEM, radius=RADIUS, cell_size=1.0)
print("Done.")

In [ ]:
# Visualise the results
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"Synthetic Bathymetry and Openness (search radius = {RADIUS} cells)",
             fontsize=13, fontweight='bold')

# --- Panel 1: Bathymetry DEM ---
im0 = axes[0].imshow(DEM, origin='lower', cmap='terrain_r')
plt.colorbar(im0, ax=axes[0], label='Depth / Elevation (m)')
axes[0].set_title("Synthetic DEM\n(high=seamount, low=pockmark)")
axes[0].set_xlabel("Column")
axes[0].set_ylabel("Row")

# Mark the seamount centre and pockmark centre
axes[0].plot(size//2, size//2, 'r^', ms=10, label='Seamount centre')
px = int((1.8 + 3) / 6 * size)
py = int((1.8 + 3) / 6 * size)
axes[0].plot(px, py, 'bv', ms=10, label='Pockmark centre')
axes[0].legend(fontsize=8)

# --- Panel 2: Negative Openness (highlights bathymetric HIGHS) ---
im1 = axes[1].imshow(NO, origin='lower', cmap='RdYlGn',
                     vmin=np.nanpercentile(NO, 2), vmax=np.nanpercentile(NO, 98))
plt.colorbar(im1, ax=axes[1], label='Negative Openness Ψ⁻ (radians)')
axes[1].set_title("Negative Openness (Ψ⁻)\nLow values → Bathymetric Highs")
axes[1].set_xlabel("Column")

# --- Panel 3: Positive Openness (highlights bathymetric LOWS) ---
im2 = axes[2].imshow(PO, origin='lower', cmap='RdYlGn',
                     vmin=np.nanpercentile(PO, 2), vmax=np.nanpercentile(PO, 98))
plt.colorbar(im2, ax=axes[2], label='Positive Openness Ψ⁺ (radians)')
axes[2].set_title("Positive Openness (Ψ⁺)\nLow values → Bathymetric Lows")
axes[2].set_xlabel("Column")

plt.tight_layout()
plt.savefig("openness_results.png", dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

In [ ]:
# ---------------------------------------------------------------
# Demonstrate thresholding — as done in GA-SaMMT
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Thresholding Openness to Detect Features", fontsize=13, fontweight='bold')

# GA-SaMMT uses: threshold = mean - c * std
# Low NO → high feature
c_large = 1.0   # NO STD Scale Large
no_threshold = np.nanmean(NO) - c_large * np.nanstd(NO)
high_mask = (NO < no_threshold)

axes[0].imshow(DEM, origin='lower', cmap='terrain_r', alpha=0.7)
axes[0].imshow(np.where(high_mask, 1, np.nan), origin='lower',
               cmap='Reds', alpha=0.7, vmin=0, vmax=1)
axes[0].set_title(f"Detected Bathymetric HIGHS\n(NO < mean - {c_large}σ = {no_threshold:.3f} rad)")
axes[0].set_xlabel("Column")
axes[0].set_ylabel("Row")

# Low PO → low feature
c_large_po = 1.5   # PO STD Scale Large
po_threshold = np.nanmean(PO) - c_large_po * np.nanstd(PO)
low_mask = (PO < po_threshold)

axes[1].imshow(DEM, origin='lower', cmap='terrain_r', alpha=0.7)
axes[1].imshow(np.where(low_mask, 1, np.nan), origin='lower',
               cmap='Blues', alpha=0.7, vmin=0, vmax=1)
axes[1].set_title(f"Detected Bathymetric LOWS\n(PO < mean - {c_large_po}σ = {po_threshold:.3f} rad)")
axes[1].set_xlabel("Column")

plt.tight_layout()
plt.savefig("openness_thresholding.png", dpi=120, bbox_inches='tight')
plt.show()
print("\nStatistics:")
print(f"  NO: mean={np.nanmean(NO):.3f}, std={np.nanstd(NO):.3f}, threshold={no_threshold:.3f} rad")
print(f"  PO: mean={np.nanmean(PO):.3f}, std={np.nanstd(PO):.3f}, threshold={po_threshold:.3f} rad")
print(f"  High-feature cells detected: {high_mask.sum()} / {high_mask.size}")
print(f"  Low-feature cells detected:  {low_mask.sum()} / {low_mask.size}")

## 9. Effect of Search Radius on Detection Scale

Changing the **search radius** fundamentally changes which features are detected. Let's compare small vs. large radii:

In [ ]:
# Compare radius=4 vs radius=12
print("Computing openness for radius=4...")
PO_small, NO_small = compute_openness(DEM, radius=4)
print("Computing openness for radius=12...")
PO_large, NO_large = compute_openness(DEM, radius=12)
print("Done.")

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle("Effect of Search Radius on Negative Openness (NO)", fontsize=13, fontweight='bold')

vmin = min(np.nanpercentile(NO_small, 2), np.nanpercentile(NO_large, 2))
vmax = max(np.nanpercentile(NO_small, 98), np.nanpercentile(NO_large, 98))

im_s = axes[0, 0].imshow(NO_small, origin='lower', cmap='RdYlGn', vmin=vmin, vmax=vmax)
plt.colorbar(im_s, ax=axes[0, 0], label='NO (rad)')
axes[0, 0].set_title("Negative Openness — radius = 4 cells")

thresh_s = np.nanmean(NO_small) - 1.0 * np.nanstd(NO_small)
axes[1, 0].imshow(DEM, origin='lower', cmap='terrain_r', alpha=0.6)
axes[1, 0].imshow(np.where(NO_small < thresh_s, 1, np.nan), origin='lower',
                  cmap='Reds', alpha=0.8)
axes[1, 0].set_title(f"Detected Highs — radius = 4\n(threshold = {thresh_s:.3f} rad)")

im_l = axes[0, 1].imshow(NO_large, origin='lower', cmap='RdYlGn', vmin=vmin, vmax=vmax)
plt.colorbar(im_l, ax=axes[0, 1], label='NO (rad)')
axes[0, 1].set_title("Negative Openness — radius = 12 cells")

thresh_l = np.nanmean(NO_large) - 1.0 * np.nanstd(NO_large)
axes[1, 1].imshow(DEM, origin='lower', cmap='terrain_r', alpha=0.6)
axes[1, 1].imshow(np.where(NO_large < thresh_l, 1, np.nan), origin='lower',
                  cmap='Reds', alpha=0.8)
axes[1, 1].set_title(f"Detected Highs — radius = 12\n(threshold = {thresh_l:.3f} rad)")

for ax in axes.flat:
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

plt.tight_layout()
plt.savefig("openness_radius_comparison.png", dpi=120, bbox_inches='tight')
plt.show()

## 10. Summary and Key Takeaways

| Concept | Definition | Key use |
|---|---|---|
| **Positive Openness (PO)** | Mean zenith angle to the highest point in each radial direction | Small PO → bathymetric low (pockmark, valley) |
| **Negative Openness (NO)** | Mean zenith angle to the lowest point in each radial direction | Small NO → bathymetric high (mound, ridge) |
| **Search radius** | Controls spatial scale of detection | Small radius = fine features; large radius = broad features |
| **STD threshold** | `mean - c×std` applied to NO or PO | Adjusting `c` controls sensitivity |

### Practical workflow in GA-SaMMT

1. **Compute** NO or PO from bathymetric grid (`BathymetricHigh.pyt` or `BathymetricLow.pyt`).
2. **Threshold** the raster using `mean - c × std` to create a binary mask of candidate cells.
3. **Vectorise** the mask into polygons.
4. **Filter** by minimum area threshold (removes noise).
5. **Add attributes** (shape, topographic, profile) using `AddAttributes.pyt`.
6. **Classify** each polygon into a morphological type (mound, ridge, pockmark, etc.) using `ClassificationFeature.pyt`.

### Openness vs. TPI: when to use which?

- Use **TPI** when working with large datasets and speed matters, or when features are broadly defined by local relief.
- Use **Openness** when features have a strong angular geometry (circular pits, conical mounds), or when the terrain has complex asymmetric shapes.
- **Combine both** for maximum robustness — the GA-SaMMT `TPI LMI` tool also combines TPI with Local Moran's I for an additional spatial autocorrelation filter.